# Module 2 | Class 4 — Pandas for Data Analysis
**Student:** Fuzaylbek  
**Topics:** DataFrames · Filtering · GroupBy · Merging  
**Dataset:** Iris (loaded from URL — no download needed in Colab)

In [ ]:
import pandas as pd
import numpy as np

---
## Section 1 — Series and DataFrames

### Task 1.1 — Recreate the lecture DataFrame and explore its structure

In [ ]:
# Recreate the exact DataFrame from the lecture
df = pd.DataFrame({
    "name": ["Ali", "Zara", "Jasur"],
    "age":  [22, 28, 31],
    "city": ["Tashkent", "Samarkand", "Bukhara"]
})

print("Original DataFrame:")
print(df)

In [ ]:
# Q1: What does df["age"] return — Series or DataFrame?
age_col = df["age"]
print("Type of df['age']:", type(age_col))
print(age_col)
print()

In [ ]:
# Q2: What does df[["name", "age"]] return?
name_age = df[["name", "age"]]
print("Type of df[['name', 'age']]:", type(name_age))
print(name_age)
print()

In [ ]:
# Q3: Print the index of df — what is it by default?
print("df.index:", df.index)

**Answers:**

- `df["age"]` returns a **Series** — single-bracket selection of one column always produces a one-dimensional Series.
- `df[["name", "age"]]` returns a **DataFrame** — double-bracket selection (passing a list of column names) always produces a two-dimensional DataFrame, even when only one column is selected this way.
- `df.index` by default is **`RangeIndex(start=0, stop=3, step=1)`** — pandas assigns integer labels 0, 1, 2, ... automatically unless you specify otherwise.

### Task 1.2 — Add three new rows and reset the index

In [ ]:
# Three additional rows
new_rows = pd.DataFrame({
    "name": ["Malika", "Bobur", "Dilnoza"],
    "age":  [25, 33, 29],
    "city": ["Namangan", "Fergana", "Andijan"]
})

# Concatenate and reset index so it stays 0..n
df = pd.concat([df, new_rows], ignore_index=True)

print("Extended DataFrame (6 rows):")
print(df)
print("\nNew index:", df.index.tolist())

---
## Section 2 — Loading and Exploring

### Task 2.1 — Load a public CSV dataset

We use the **Iris** dataset — a small, clean, well-known public dataset with 150 rows and 5 columns (four numeric flower measurements + one species label). It is loaded directly from a public URL so no file upload is needed in Google Colab.

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
iris = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", iris.shape)

> **Note on `parse_dates`:** Iris has no date column, so `parse_dates` is not applicable here. If your dataset had a date column (e.g. `"date"`), you would reload it as:
> ```python
> pd.read_csv("file.csv", parse_dates=["date"])
> ```
> This tells pandas to parse that column as `datetime64` instead of plain text, enabling time-series operations like resampling and date arithmetic.

### Task 2.2 — Five inspection commands

In [ ]:
# 1) head() — first 5 rows
iris.head()

**What I learned from `head()`:** The dataset has four decimal-valued measurement columns (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) and one string label column (`species`). All values look well-formed with no obvious anomalies in the first five rows.

In [ ]:
# 2) shape — (rows, columns)
print("Shape:", iris.shape)

**What I learned from `shape`:** The dataset has exactly 150 rows and 5 columns — the canonical Iris size (50 samples per species × 3 species), confirming the file loaded completely.

In [ ]:
# 3) info() — column data types and non-null counts
iris.info()

**What I learned from `info()`:** All four numeric columns are `float64` and `species` is `object` (string). Every column shows 150 non-null entries — there are zero missing values and no imputation is needed before analysis.

In [ ]:
# 4) describe() — summary statistics for numeric columns
iris.describe()

**What I learned from `describe()`:** `petal_length` has the widest range (min = 1.0, max = 6.9) and the highest standard deviation (~1.77), meaning it varies the most across species and is likely the most discriminative feature for classification.

In [ ]:
# 5) isna().sum() — count of missing values per column
iris.isna().sum()

**What I learned from `isna().sum()`:** Every column returns 0. The Iris dataset is perfectly clean, so we can move straight to analysis without any data-cleaning steps.

---
## Section 3 — Filtering, GroupBy, Merging

### Task 3.1 — Selection

In [ ]:
# Select one column as a Series
petal_len = iris["petal_length"]
print("Type:", type(petal_len))   # pandas.core.series.Series
print(petal_len.head())

In [ ]:
# Select two columns as a DataFrame
two_cols = iris[["sepal_length", "species"]]
print("Type:", type(two_cols))    # pandas.core.frame.DataFrame
print(two_cols.head())

In [ ]:
# Use df.loc[] to select a row by its label (label = 10)
print("iris.loc[10] — row with label 10:")
print(iris.loc[10])

In [ ]:
# Use df.iloc[] to select the first row by integer position
print("iris.iloc[0] — first row by position:")
print(iris.iloc[0])

**`loc` vs `iloc` — Quick Check #1:**

`loc` selects by **label** (the actual value in the index). `iloc` selects by **integer position** (0-based, like a Python list). When the index is the default `RangeIndex`, labels and positions coincide, so the difference is invisible. It becomes critical once you have a custom index — for example after setting a date column as the index: `df.loc["2024-01-15"]` works; `df.iloc["2024-01-15"]` raises a `TypeError`.

### Task 3.2 — Boolean Filtering

In [ ]:
# Filter 1: sepal_length >= 6.0  AND  species == "virginica"
filter1 = iris[(iris["sepal_length"] >= 6.0) & (iris["species"] == "virginica")]
print("Filter 1 — virginica rows with sepal_length >= 6.0:", filter1.shape)
filter1.head()

In [ ]:
# Filter 2: petal_width <= 0.5  OR  species == "setosa"
filter2 = iris[(iris["petal_width"] <= 0.5) | (iris["species"] == "setosa")]
print("Filter 2 — petal_width <= 0.5 OR setosa:", filter2.shape)
filter2.head()

**Why must we use `&` instead of `and`, and why are parentheses required? (Quick Check #2)**

Python's built-in `and` keyword is designed for **scalar** boolean logic — it evaluates two single `True`/`False` values and short-circuits. When applied to a pandas Series, Python cannot reduce an entire Series to a single truth value and raises:
```
ValueError: The truth value of a Series is ambiguous.
```

The `&` operator (bitwise AND) is **overloaded** by pandas to work **element-wise** across two boolean Series of the same length, producing a new boolean Series — which is exactly what row-level filtering needs. Similarly, `|` is used for element-wise OR.

**Parentheses** are mandatory because of Python's operator precedence: `&` binds *more tightly* than the comparison operators (`>=`, `==`, etc.). Without parentheses, Python would parse `iris["sepal_length"] >= 6.0 & iris["species"] == "virginica"` as `iris["sepal_length"] >= (6.0 & iris["species"]) == "virginica"` — a nonsensical expression that raises an error. Wrapping each condition in `()` forces Python to evaluate the comparisons first, then combine the resulting boolean Series with `&`.

### Task 3.3 — GroupBy: Split-Apply-Combine

In [ ]:
# Pattern 1: single aggregation — mean petal_length per species
result1 = iris.groupby("species")["petal_length"].mean()

print("Return type:", type(result1))   # pandas.core.series.Series
print()
print(result1)

**What type does the first call return? (Quick Check #3)**

`df.groupby("category")["numeric"].mean()` returns a **pandas Series**. The group keys (`setosa`, `versicolor`, `virginica`) become the index, and the aggregated values are the data. Because we selected a single column before aggregating, the result is one-dimensional — a Series, not a DataFrame.

In [ ]:
# Pattern 2: agg() with multiple aggregation functions on different columns
result2 = iris.groupby("species").agg({
    "petal_length": "mean",   # average petal length per species
    "sepal_width":  "count"   # number of records per species (should be 50 each)
})

print("Return type:", type(result2))   # pandas.core.frame.DataFrame
print()
print(result2)

> When `agg({...})` operates on multiple columns, the result is a **DataFrame** — each key in the dict becomes a column in the output, and the group keys form the index.

### Task 3.4 — Merging

In [ ]:
# Build a small lookup DataFrame manually: species -> region + rarity score
lookup_df = pd.DataFrame({
    "species":      ["setosa", "versicolor", "virginica"],
    "region":       ["Arctic",  "Temperate",  "Tropical"],
    "rarity_score": [3,         2,            1]          # 1 = rarest
})

print("Lookup DataFrame:")
print(lookup_df)

In [ ]:
# Left join: keep all iris rows, attach lookup columns where species matches
merged = pd.merge(iris, lookup_df, on="species", how="left")

print("Merged shape:", merged.shape)          # still 150 rows
print("Columns:", merged.columns.tolist())
print()
merged.head(10)

In [ ]:
# Demonstrate what happens when a left-table row has NO match in the lookup.
# Add one row with a species that does not exist in lookup_df.
extra_row = pd.DataFrame([{
    "sepal_length": 5.0, "sepal_width": 3.0,
    "petal_length": 1.5, "petal_width": 0.3,
    "species": "unknown_species"
}])

iris_with_extra = pd.concat([iris, extra_row], ignore_index=True)
merged_demo = pd.merge(iris_with_extra, lookup_df, on="species", how="left")

print("Last row (unmatched species):")
print(merged_demo.tail(1).to_string())
print()
print("NaN count in 'region':", merged_demo["region"].isna().sum())
print("NaN count in 'rarity_score':", merged_demo["rarity_score"].isna().sum())

**What happens to rows in the left table that have no match in the lookup? (Quick Check #4)**

With `how="left"`, pandas preserves **every row from the left DataFrame** regardless of whether a matching key exists in the right (lookup) table. For rows with no match, all columns that would have come from the right table are filled with **`NaN`** (Not a Number — pandas' sentinel for a missing value).

In the demo above, `"unknown_species"` is not in `lookup_df`, so its `region` and `rarity_score` fields are `NaN`, while the original flower measurements remain intact. This means a left join never loses data from the left table but signals missing enrichment data with `NaN`.

---
## Quick Check Summary

| # | Question | Answer |
|---|----------|--------|
| 1 | `loc` vs `iloc` | `loc` selects by **label** (index value); `iloc` selects by **integer position** (0-based). They differ when the index is non-default (strings, dates, etc.). |
| 2 | `&` vs `and`, and parentheses | `and` is scalar-only and raises `ValueError` on a Series. `&` applies element-wise across two boolean Series. Parentheses are required because `&` has higher precedence than `>=` / `==`. |
| 3 | GroupBy return type | `.groupby(col)[col].mean()` returns a **Series**. `.groupby(col).agg({...})` on multiple columns returns a **DataFrame**. |
| 4 | Left join, no match | The left row is **kept**; columns from the right table are filled with **`NaN`**. No data is lost from the left side. |
| 5 | Large CSV strategy | Use `pd.read_csv(..., chunksize=N)` to read in chunks and process iteratively, or pass `usecols=[...]` to load only the columns you need — both approaches avoid loading the full file into memory at once. |